In [4]:
# ============================================================
# ZapKart Dark Store Intelligence
# Notebook 4: SQL Analysis
# ============================================================

import pandas as pd
import numpy as np
import psycopg2
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

PROCESSED_PATH = "D:\Projects\End-to-end projects\9. ZapKart Dark-Store Intelligence\Data\Processed" + "\\"

# Use psycopg2 connection string directly
# This method handles special characters like @ in passwords safely
engine = create_engine(
    "postgresql+psycopg2://",
    creator=lambda: psycopg2.connect(
        host     = "localhost",
        port     = 5432,
        dbname   = "zapkart_db",
        user     = "postgres",
        password = "NewStrongPassword@123"
    )
)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT version()"))
    print("✅ Connected to PostgreSQL")
    print(f"   {result.fetchone()[0]}")

✅ Connected to PostgreSQL
   PostgreSQL 17.6 on x86_64-windows, compiled by msvc-19.44.35213, 64-bit


In [5]:
# ============================================================
# Load all cleaned CSVs into PostgreSQL tables
# ============================================================

tables = {
    'dim_stores':           'dim_stores_clean.csv',
    'dim_skus':             'dim_skus_clean.csv',
    'dim_customers':        'dim_customers_clean.csv',
    'dim_pickers':          'dim_pickers_clean.csv',
    'fact_orders':          'fact_orders_clean.csv',
    'fact_order_items':     'fact_order_items_clean.csv',
    'fact_picker_activity': 'fact_picker_activity_clean.csv',
    'fact_substitutions':   'fact_substitutions_clean.csv'
}

print("Loading tables into PostgreSQL...")
print("=" * 55)

for table_name, file_name in tables.items():
    df = pd.read_csv(f"{PROCESSED_PATH}{file_name}")
    df.to_sql(table_name, engine,
              if_exists = 'replace',
              index     = False,
              chunksize = 10000)
    print(f"  ✅ {table_name:<30} {len(df):>10,} rows loaded")

print("=" * 55)
print("🎉 All 8 tables loaded into zapkart_db")

Loading tables into PostgreSQL...
  ✅ dim_stores                             10 rows loaded
  ✅ dim_skus                              499 rows loaded
  ✅ dim_customers                      47,610 rows loaded
  ✅ dim_pickers                            80 rows loaded
  ✅ fact_orders                       453,036 rows loaded
  ✅ fact_order_items                2,718,037 rows loaded
  ✅ fact_picker_activity              412,015 rows loaded
  ✅ fact_substitutions                 54,396 rows loaded
🎉 All 8 tables loaded into zapkart_db


In [6]:
# ============================================================
# Verify all tables loaded correctly
# ============================================================

verify_sql = """
SELECT
    table_name,
    pg_size_pretty(pg_total_relation_size(
        quote_ident(table_name))) AS size
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

with engine.connect() as conn:
    result = pd.read_sql(verify_sql, conn)

print("Tables in zapkart_db:")
print(result.to_string(index=False))

Tables in zapkart_db:
          table_name    size
       dim_customers 7848 kB
         dim_pickers   48 kB
            dim_skus  120 kB
          dim_stores   16 kB
    fact_order_items  406 MB
         fact_orders  139 MB
fact_picker_activity   70 MB
  fact_substitutions 8560 kB
